In [5]:
import pandas as pd
import numpy as np

In [6]:
ledger = pd.read_csv('/content/ledger.csv')
gateway = pd.read_csv('/content/gateway.csv')

In [7]:
##check null and dupicates
print("Null values in Ledger:")
print(ledger.isnull().sum())

print("\nDuplicate rows in Ledger:")
print(ledger.duplicated().sum())

print("\nNull values in Gateway:")
print(gateway.isnull().sum())

print("\nDuplicate rows in Gateway:")
print(gateway.duplicated().sum())

Null values in Ledger:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Duplicate rows in Ledger:
0

Null values in Gateway:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Duplicate rows in Gateway:
0


In [8]:
##missing in gateway
ledger_ids = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

id_missing_in_gateway = ledger_ids - gateway_ids

print(f"Count of id_missing_in_gateway: {len(id_missing_in_gateway)}")

print("\nMissing in gateway:")
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]
display(missing_in_gateway)

Count of id_missing_in_gateway: 2

Missing in gateway:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


In [9]:
##missing in ledger
gateway_ids = set(gateway['transaction_id'])
ledger_ids = set(ledger['transaction_id'])

id_missing_in_ledger = gateway_ids - ledger_ids

print(f"count of id_missing_in_ledger: {len(id_missing_in_ledger)}")

print("\nMissing in ledger:")
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])]
display(missing_in_ledger)

count of id_missing_in_ledger: 1

Missing in ledger:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,2026-03-05,M003,1800.0,success,Card


In [10]:
##amount mismatches
merged_amount_comparison_df = pd.merge(
ledger[['transaction_id', 'amount_usd']],
    gateway[['transaction_id', 'amount_usd']],
    on='transaction_id',
    suffixes=('_ledger', '_gateway'),
    how='inner'
)

amount_mismatches = merged_amount_comparison_df[
    (merged_amount_comparison_df['amount_usd_ledger'] != merged_amount_comparison_df['amount_usd_gateway'])
]

print(f"Count of amount mismatches: {len(amount_mismatches)}")
print("\nAmount mismatches:")
display(amount_mismatches)

Count of amount mismatches: 2

Amount mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway
1,R002,850.0,900.0
6,R008,640.0,600.0


In [11]:
##status mismatches
merged_status_comparison_df = pd.merge(
    ledger[['transaction_id', 'status']],
    gateway[['transaction_id', 'status']],
    on='transaction_id',
    suffixes=('_ledger', '_gateway'),
    how='inner'
)

status_mismatches = merged_status_comparison_df[
    (merged_status_comparison_df['status_ledger'] != merged_status_comparison_df['status_gateway'])
]

print(f"Count of status mismatches: {len(status_mismatches)}")
print("\nStatus mismatches:")
display(status_mismatches)

Count of status mismatches: 1

Status mismatches:


,transaction_id,status_ledger,status_gateway
3,R005,success,failed


In [12]:
report_missing_gateway_df = pd.DataFrame({
    'transaction_id': missing_in_gateway['transaction_id'],
    'ledger_amount': missing_in_gateway['amount_usd'],
    'gateway_amount': np.nan,  # Amount is missing in gateway
    'difference': missing_in_gateway['amount_usd'], # The entire ledger amount is the difference
    'issue_type': 'Missing in Gateway'
})

report_missing_ledger_df = pd.DataFrame({
    'transaction_id': missing_in_ledger['transaction_id'],
    'ledger_amount': np.nan, # Amount is missing in ledger
    'gateway_amount': missing_in_ledger['amount_usd'],
    'difference': missing_in_ledger['amount_usd'], # The entire gateway amount is the difference
    'issue_type': 'Missing in Ledger'
})

report_amount_mismatch_df = pd.DataFrame({
    'transaction_id': amount_mismatches['transaction_id'],
    'ledger_amount': amount_mismatches['amount_usd_ledger'],
    'gateway_amount': amount_mismatches['amount_usd_gateway'],
    'difference': amount_mismatches['amount_usd_gateway'] - amount_mismatches['amount_usd_ledger'],
    'issue_type': 'Amount Mismatch'
})

reconciliation_report = pd.concat([report_missing_gateway_df, report_missing_ledger_df, report_amount_mismatch_df])

reconciliation_report['abs_difference'] = reconciliation_report['difference'].abs()
reconciliation_report = reconciliation_report.sort_values(by='abs_difference', ascending=False)

reconciliation_report = reconciliation_report.reset_index(drop=True).drop(columns=['abs_difference'])

print("Count of issues per type:")
print(reconciliation_report['issue_type'].value_counts())

print("\nFinal Reconciliation Report:")
display(reconciliation_report)

Count of issues per type:
issue_type
Missing in Gateway    2
Amount Mismatch       2
Missing in Ledger     1
Name: count, dtype: int64

Final Reconciliation Report:


,transaction_id,ledger_amount,gateway_amount,difference,issue_type
0,R010,2500.0,NaN,2500.0,Missing in Gateway
1,R004,2100.0,NaN,2100.0,Missing in Gateway
2,R011,NaN,1800.0,1800.0,Missing in Ledger
3,R002,850.0,900.0,50.0,Amount Mismatch
4,R008,640.0,600.0,-40.0,Amount Mismatch


In [13]:
##JSON NORMALIZATION

In [14]:
import json

with open('/content/api_response_sample.json', 'r') as f:
    api_response = json.load(f)

In [15]:
normalized_df = pd.json_normalize(api_response['batches'])
display(normalized_df.head())

,batch_id,settlements,merchant.merchant_id,merchant.merchant_name,merchant.region
0,B001,"[{'settlement_id': 'S001', 'amount_usd': 1520....",M001,Alpha Mart,APAC
1,B002,"[{'settlement_id': 'S004', 'amount_usd': 2100....",M004,Delta Travels,US


In [16]:
def clean_col_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:

        new_col = col.replace('.', '_')

        new_col = new_col.lower()
        new_cols.append(new_col)
    df.columns = new_cols
    return df

normalized_df = clean_col_names(normalized_df)
display(normalized_df.head())

,batch_id,settlements,merchant_merchant_id,merchant_merchant_name,merchant_region
0,B001,"[{'settlement_id': 'S001', 'amount_usd': 1520....",M001,Alpha Mart,APAC
1,B002,"[{'settlement_id': 'S004', 'amount_usd': 2100....",M004,Delta Travels,US


In [17]:
print("Original Dtypes:")
print(normalized_df.dtypes)

if 'transaction_date' in normalized_df.columns:
    normalized_df['transaction_date'] = pd.to_datetime(normalized_df['transaction_date'], errors='coerce')
if 'payment_details_timestamp' in normalized_df.columns:
    normalized_df['payment_details_timestamp'] = pd.to_datetime(normalized_df['payment_details_timestamp'], errors='coerce')

print("\nNew Dtypes after date conversion:")
print(normalized_df.dtypes)
display(normalized_df.head())

Original Dtypes:
batch_id                  object
settlements               object
merchant_merchant_id      object
merchant_merchant_name    object
merchant_region           object
dtype: object

New Dtypes after date conversion:
batch_id                  object
settlements               object
merchant_merchant_id      object
merchant_merchant_name    object
merchant_region           object
dtype: object


,batch_id,settlements,merchant_merchant_id,merchant_merchant_name,merchant_region
0,B001,"[{'settlement_id': 'S001', 'amount_usd': 1520....",M001,Alpha Mart,APAC
1,B002,"[{'settlement_id': 'S004', 'amount_usd': 2100....",M004,Delta Travels,US


In [18]:
output_filename = 'normalized_api_transactions.csv'
normalized_df.to_csv(output_filename, index=False)
print(f"Normalized data saved to {output_filename}")

Normalized data saved to normalized_api_transactions.csv
